In [1]:
import numpy as np # loading data
import torch
from torch.utils.data import DataLoader,TensorDataset # Tensor dataset and construct dataloader
import torch.optim as optim # Optimization function
import torch.nn as nn # Call layers and construct model
from torch.optim.lr_scheduler import ReduceLROnPlateau # Adjust learning rate during training
# from torchmetrics.functional import precision_recall
from sklearn import metrics # Model performance metrics
import torchvision.models as models # Import pretrained model
from torchvision.models import ResNet18_Weights
import pickle
import pandas as pd
from skorch import NeuralNetClassifier # Use PyTorch Models in scikit-learn
from sklearn.model_selection import GridSearchCV # Grid Search

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)

cuda


# Data

In [3]:
with open('C:/Users/101194208/Desktop/Die-Cast Ensemble/Data/data_oversample_usingViT.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

# Access the loaded variables
X_train = loaded_data['var1']
X_val = loaded_data['var2']
X_test = loaded_data['var3']
y_train = loaded_data['var4']
y_val = loaded_data['var5']
y_test = loaded_data['var6']

X_train = torch.Tensor(X_train)
y_train = torch.LongTensor(y_train)

# Model Construction

In [4]:
# Load a pre-trained ResNet-18 model using weights
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Replace the last fully connected layer for three-class classification
num_classes = 3  # Change from 2 to 3 for three classes

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, num_classes)  # Updated output layer for 3 classes
) 
# Do NOT apply nn.Softmax(dim=1) here, as CrossEntropyLoss includes it implicitly

# Create Model with skorch

In [5]:
model = NeuralNetClassifier(
    models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1),
    criterion = nn.CrossEntropyLoss(),
    optimizer = optim.Adam,
    verbose = False,
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# Grid Search

In [6]:
param_grid = {
    'batch_size': [32, 64],
    'max_epochs': [20, 30, 40],
    'optimizer__lr': [0.0001, 0.001, 0.01, 0.1]}

In [7]:
%%time

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3)
grid_result = grid.fit(X_train, y_train)

CPU times: total: 5h 38min 44s
Wall time: 43min 13s


In [8]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.531726 using {'batch_size': 64, 'max_epochs': 30, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.001}
0.346312 (0.018211) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.01}
0.333745 (0.000874) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.1}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.0001}
0.332921 (0.000771) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.001}
0.349197 (0.022725) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.01}
0.334569 (0.000771) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.1}
0.333539 (0.000291) with: {'batch_size': 32, 'max_epochs': 40, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 40, 'optimizer__lr': 0.001}
0.360321 (0.034537) with: {'batch_size': 32, 'max_